# 02 — Visualization & Trend Analysis

**Dataset:** Pengangguran Menurut Golongan Umur (2021–2025)  
**Source:** BPS — Badan Pusat Statistik (Statistics Indonesia)  
**Source URL:** https://www.bps.go.id/  
**Data date:** 2021 Februari – 2025 Agustus  
**Purpose:** Produce final interactive charts and static figures for reporting and Kaggle submission.

## Imports

In [1]:
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

random.seed(42)
np.random.seed(42)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('Libraries loaded.')


Libraries loaded.


## Configuration / Constants

In [2]:
DATA_DIR = (
    Path('/kaggle/input/unemployment-indonesia')
    if Path('/kaggle').exists()
    else Path('../data/processed')
)
OUTPUTS_DIR = Path('../outputs') if not Path('/kaggle').exists() else Path('/kaggle/working')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

AGE_GROUP_ORDER = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60+']
PERIOD_ORDER = ['Februari', 'Agustus']
SOURCE_NOTE = 'Source: BPS — Badan Pusat Statistik (Statistics Indonesia)'

# Plotly discrete sequence — colorblind-friendly
PLOTLY_COLORS = px.colors.qualitative.Safe

print(f'DATA_DIR: {DATA_DIR.resolve()}')
print(f'OUTPUTS_DIR: {OUTPUTS_DIR.resolve()}')

DATA_DIR: C:\Users\Admin\project\Data Analysis\unemployment-analysis\data\processed
OUTPUTS_DIR: C:\Users\Admin\project\Data Analysis\unemployment-analysis\outputs


## Data Loading

Load the consolidated long-format CSV produced by notebook 01_eda.ipynb.

In [3]:
df = pd.read_csv(DATA_DIR / 'unemployment_combined.csv')

age_cat = pd.CategoricalDtype(categories=AGE_GROUP_ORDER, ordered=True)
period_cat = pd.CategoricalDtype(categories=PERIOD_ORDER, ordered=True)
df['age_group'] = df['age_group'].astype(str).astype(age_cat)
df['period'] = df['period'].astype(str).astype(period_cat)
df.sort_values(['year', 'period', 'age_group'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Loaded: {df.shape}')
df.head()

Loaded: (100, 6)


,year,period,age_group,pernah_bekerja,tidak_pernah_bekerja,jumlah
0,2021,Februari,15-19,281236,931164,1212400
1,2021,Februari,20-24,1338570,1270769,2609339
2,2021,Februari,25-29,1111006,423633,1534639
3,2021,Februari,30-34,654139,152522,806661
4,2021,Februari,35-39,518758,99728,618486


In [4]:
# Build stable time labels for x-axis ordering
period_abbr = {'Februari': 'Feb', 'Agustus': 'Aug'}
period_sort = {'Februari': 0, 'Agustus': 1}

df['time_label'] = df['year'].astype(str) + '-' + df['period'].astype(str).map(period_abbr)
df['_sort_key'] = df['year'].astype(int) * 10 + df['period'].astype(str).map(period_sort)

time_order = (
    df[['time_label', '_sort_key']].drop_duplicates()
    .sort_values('_sort_key')['time_label']
    .tolist()
)
print('Time axis order:', time_order)

Time axis order: ['2021-Feb', '2021-Aug', '2022-Feb', '2022-Aug', '2023-Feb', '2023-Aug', '2024-Feb', '2024-Aug', '2025-Feb', '2025-Aug']


## Data Cleaning & Validation

The processed CSV from notebook 01 is already in the canonical long format. Here we
re-apply categorical dtypes for correct sort order (done inline in the Data Loading cell),
re-derive the `time_label` / `_sort_key` helper columns used across all charts, and
run a quick sanity check to confirm shape, no duplicates, no nulls, and the invariant
`pernah_bekerja + tidak_pernah_bekerja == jumlah` holds before any chart is produced.


In [6]:
# Validate the loaded dataset before charting — shape, duplicates, and invariant checks
assert df.shape == (100, 8), f"Expected (100, 6), got {df.shape}"
assert not df.duplicated(subset=['year', 'period', 'age_group']).any(), "Duplicate rows detected"
assert (df['pernah_bekerja'] + df['tidak_pernah_bekerja'] == df['jumlah']).all(), \
    "Accounting invariant broken: pernah_bekerja + tidak_pernah_bekerja != jumlah"
assert df.isnull().sum().sum() == 0, "Unexpected null values detected"
print(f"Validation passed — {df.shape[0]} rows, {df.shape[1]} columns, no nulls, no duplicates.")
df.sample(5, random_state=42)


Validation passed — 100 rows, 8 columns, no nulls, no duplicates.


,year,period,age_group,pernah_bekerja,tidak_pernah_bekerja,jumlah,time_label,_sort_key
83,2025,Februari,30-34,469085,150715,619800,2025-Feb,20250
53,2023,Agustus,30-34,449519,160449,609968,2023-Aug,20231
70,2024,Agustus,15-19,181907,1250053,1431960,2024-Aug,20241
45,2023,Februari,40-44,528134,151937,680071,2023-Feb,20230
44,2023,Februari,35-39,502226,168040,670266,2023-Feb,20230


## Visualization

All charts use **Plotly** for interactive display in Kaggle / Jupyter. Each figure answers a
specific analytical question defined in the project scope, covers all 10 survey periods
(2021-Feb through 2025-Aug), and is exported to `outputs/` as a standalone HTML file for
offline sharing. Every chart includes a title, labeled axes with units, a legend where
multiple series are shown, and a source note.


### Chart 1: Unemployment Trend by Age Group

**Why:** This multi-line chart answers whether the 20–24 cohort is persistently the most affected
and reveals each cohort's individual recovery (or lack thereof) across all 10 survey periods.
It makes the post-pandemic decline visible and highlights cohorts that plateaued vs. those that
continued to improve.


In [7]:
fig1 = px.line(
    df.sort_values('_sort_key'),
    x='time_label',
    y='jumlah',
    color='age_group',
    category_orders={'time_label': time_order, 'age_group': AGE_GROUP_ORDER},
    markers=True,
    color_discrete_sequence=PLOTLY_COLORS,
)
fig1.update_layout(
    title='Unemployment by Age Group Over Time (2021–2025)',
    xaxis_title='Survey Period',
    yaxis_title='Jumlah Pengangguran (Orang)',
    legend_title='Age Group',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig1.write_html(str(OUTPUTS_DIR / 'interactive_trend.html'))
fig1.show()

### Chart 2: February vs. August Total Unemployment per Year

**Why:** A side-by-side grouped bar chart directly answers whether there is a consistent seasonal
pattern between the two survey periods. If August is systematically higher than February
(driven by mid-year graduation waves), the direction and magnitude of that gap changes
policy relevance — e.g., distinguishing structural unemployment from cyclical/seasonal flows.


In [8]:
agg = df.groupby(['year', 'period'], observed=True)['jumlah'].sum().reset_index()
agg['period'] = agg['period'].astype(str)

fig2 = px.bar(
    agg,
    x='year',
    y='jumlah',
    color='period',
    barmode='group',
    category_orders={'period': PERIOD_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
)
fig2.update_layout(
    title='Total Unemployment: February vs. August per Year (2021–2025)',
    xaxis_title='Year',
    yaxis_title='Jumlah Pengangguran (Orang)',
    legend_title='Survey Period',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig2.write_html(str(OUTPUTS_DIR / 'feb_vs_aug.html'))
fig2.show()

### Chart 3: Unemployment Heatmap (Age Group × Survey Period)

**Why:** The heatmap encodes all 100 data points simultaneously, making cross-cohort and
cross-period patterns immediately visible. It is particularly useful for spotting the 60+
group's anomalously large inter-period swings and for confirming that the 20–24 cell is
consistently the darkest (highest count) across the entire grid.


In [7]:
pivot = df.pivot_table(index='age_group', columns='time_label', values='jumlah', aggfunc='sum')
pivot = pivot.loc[AGE_GROUP_ORDER, time_order]

fig3 = px.imshow(
    pivot,
    color_continuous_scale='YlOrRd',
    text_auto=True,
    aspect='auto',
)
fig3.update_layout(
    title='Unemployment Heatmap: Age Group × Survey Period (2021–2025)',
    xaxis_title='Survey Period',
    yaxis_title='Age Group',
    coloraxis_colorbar_title='Orang',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.12, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig3.write_html(str(OUTPUTS_DIR / 'heatmap.html'))
fig3.show()

### Chart 4: Pernah vs. Tidak Pernah Bekerja Composition by Age Group

**Why:** A stacked bar chart averaged across all periods reveals whether youth cohorts skew
toward first-time job seekers (`Tidak Pernah Bekerja`) while older cohorts skew toward
displaced workers (`Pernah Bekerja`). This compositional view informs whether unemployment
policy should target re-employment programs (for older displaced cohorts) or entry-level
job creation and vocational training (for younger never-employed cohorts).


In [9]:
comp = df.groupby('age_group', observed=True)[['pernah_bekerja', 'tidak_pernah_bekerja']].mean().reset_index()
comp['age_group'] = comp['age_group'].astype(str)
comp_melt = comp.melt(id_vars='age_group', var_name='category', value_name='mean_count')
label_map = {'pernah_bekerja': 'Pernah Bekerja', 'tidak_pernah_bekerja': 'Tidak Pernah Bekerja'}
comp_melt['category'] = comp_melt['category'].map(label_map)

fig4 = px.bar(
    comp_melt,
    x='age_group',
    y='mean_count',
    color='category',
    barmode='stack',
    category_orders={'age_group': AGE_GROUP_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
)
fig4.update_layout(
    title='Composition: Previously Employed vs. Never Employed by Age Group (Average)',
    xaxis_title='Age Group',
    yaxis_title='Mean Jumlah Pengangguran (Orang)',
    legend_title='Employment History',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig4.write_html(str(OUTPUTS_DIR / 'composition_by_age.html'))
fig4.show()

### Chart 5: Year-over-Year Change in Total Unemployment

**Why:** Percentage change isolates the rate of improvement (or deterioration) from absolute
magnitude, making it easy to identify which years had the sharpest policy-relevant drops.
The dashed zero-line clearly marks the boundary between improvement and worsening. Bars
approaching 0% in 2024–2025 are the key signal that cyclical recovery has largely run its
course and remaining unemployment is structural.


In [10]:
annual = df.groupby(['year', 'period'], observed=True)['jumlah'].sum().reset_index()
annual['period_str'] = annual['period'].astype(str)
annual.sort_values(['period_str', 'year'], inplace=True)
annual['yoy_pct'] = annual.groupby('period_str')['jumlah'].pct_change() * 100

fig5 = px.bar(
    annual.dropna(subset=['yoy_pct']),
    x='year',
    y='yoy_pct',
    color='period_str',
    barmode='group',
    category_orders={'period_str': PERIOD_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
)
fig5.add_hline(y=0, line_dash='dash', line_color='black', line_width=1)
fig5.update_layout(
    title='Year-over-Year Change in Total Unemployment (%)',
    xaxis_title='Year',
    yaxis_title='YoY Change (%)',
    legend_title='Survey Period',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig5.write_html(str(OUTPUTS_DIR / 'yoy_change.html'))
fig5.show()

### Chart 6: National Total Unemployment Trend

**Why (Q2):** Chart 1 disaggregates unemployment by cohort. This companion chart collapses all
age groups into a single national aggregate line, making the post-pandemic recovery trajectory —
and any plateau — easier to read without the visual noise of 10 overlapping series. It is the
clearest answer to whether total unemployment is consistently declining or has levelled off.


In [11]:
nat_trend = (
    df.groupby(['year', 'period'], observed=True)['jumlah']
    .sum()
    .reset_index()
)
nat_trend['time_label_nat'] = nat_trend['year'].astype(str) + '-' + nat_trend['period'].astype(str).map(period_abbr)
nat_trend['_sort_key_nat'] = nat_trend['year'].astype(int) * 10 + nat_trend['period'].astype(str).map(period_sort)
nat_trend = nat_trend.sort_values('_sort_key_nat')

fig6 = px.line(
    nat_trend,
    x='time_label_nat',
    y='jumlah',
    category_orders={'time_label_nat': time_order},
    markers=True,
    color_discrete_sequence=[PLOTLY_COLORS[0]],
    title='National Total Unemployment Trend Across All Survey Periods (2021–2025)',
    labels={'time_label_nat': 'Survey Period', 'jumlah': 'Jumlah Pengangguran (Orang)'},
)
fig6.update_layout(
    xaxis_title='Survey Period',
    yaxis_title='Jumlah Pengangguran (Orang)',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig6.write_html(str(OUTPUTS_DIR / 'national_trend.html'))
fig6.show()


### Chart 7: Total Unemployment by Age Group per Year

**Why (Q1):** Chart 1 shows trends across time. This chart ranks all 10 age groups within each
individual year, making the dominance of the 20–24 cohort immediately visually obvious and
allowing year-by-year comparison of how rankings shift. It directly answers whether 20–24 is
*persistently* the most affected or whether other groups challenged that position in specific years.


In [12]:
rank_data = (
    df.groupby(['year', 'age_group'], observed=True)['jumlah']
    .sum()
    .reset_index()
)
rank_data['age_group'] = rank_data['age_group'].astype(str)

fig7 = px.bar(
    rank_data,
    x='age_group',
    y='jumlah',
    color='age_group',
    facet_col='year',
    facet_col_wrap=3,
    category_orders={'age_group': AGE_GROUP_ORDER},
    color_discrete_sequence=PLOTLY_COLORS,
    title='Total Unemployment by Age Group per Year (2021–2025)',
    labels={'age_group': 'Age Group', 'jumlah': 'Jumlah Pengangguran (Orang)'},
)
fig7.update_layout(
    yaxis_title='Jumlah Pengangguran (Orang)',
    yaxis_tickformat=',',
    showlegend=False,
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.12, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig7.update_xaxes(tickangle=45)
fig7.write_html(str(OUTPUTS_DIR / 'age_group_ranking.html'))
fig7.show()


### Chart 8: Year-over-Year Change by Age Group (%)

**Why (Q4):** Chart 5 shows the national-level YoY change. This chart breaks that down by age
group, revealing which cohorts recovered fastest (largest negative bars in 2022) and which
stalled or even worsened. It answers which age group shows the steepest improvement and whether
younger or older cohorts are leading the recovery — information invisible in the national aggregate.


In [13]:
age_annual = (
    df.groupby(['year', 'period', 'age_group'], observed=True)['jumlah']
    .sum()
    .reset_index()
)
age_annual['period_str'] = age_annual['period'].astype(str)
age_annual['age_group_str'] = age_annual['age_group'].astype(str)
age_annual.sort_values(['age_group_str', 'period_str', 'year'], inplace=True)
age_annual['yoy_pct'] = (
    age_annual.groupby(['age_group_str', 'period_str'])['jumlah'].pct_change() * 100
)
age_annual_yoy = age_annual.dropna(subset=['yoy_pct']).copy()

fig8 = px.bar(
    age_annual_yoy,
    x='age_group_str',
    y='yoy_pct',
    color='period_str',
    barmode='group',
    facet_col='year',
    facet_col_wrap=2,
    category_orders={'age_group_str': AGE_GROUP_ORDER, 'period_str': PERIOD_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
    title='Year-over-Year Change in Unemployment by Age Group (%)',
    labels={
        'age_group_str': 'Age Group',
        'yoy_pct': 'YoY Change (%)',
        'period_str': 'Survey Period',
    },
)
fig8.add_hline(y=0, line_dash='dash', line_color='black', line_width=1)
fig8.update_xaxes(tickangle=45)
fig8.update_layout(
    yaxis_title='YoY Change (%)',
    legend_title='Survey Period',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.12, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig8.write_html(str(OUTPUTS_DIR / 'yoy_by_age_group.html'))
fig8.show()


### Chart 9: Pernah Bekerja vs. Tidak Pernah Bekerja — National Totals Over Time

**Why (Q6):** Chart 4 shows the *composition by age group* (averaged). This chart instead tracks
*national totals* for each category over all 10 survey periods, asking whether re-employment of
displaced workers is outpacing first-time labor market entry. If `Pernah Bekerja` falls faster
than `Tidak Pernah Bekerja`, cyclical recovery is the dominant force; if `Tidak Pernah Bekerja`
remains flat while the other falls, the structural first-employment barrier is not being addressed.


In [14]:
nat_comp = (
    df.groupby(['year', 'period'], observed=True)[['pernah_bekerja', 'tidak_pernah_bekerja']]
    .sum()
    .reset_index()
)
nat_comp['time_label_n'] = nat_comp['year'].astype(str) + '-' + nat_comp['period'].astype(str).map(period_abbr)
nat_comp['_sort_key_n'] = nat_comp['year'].astype(int) * 10 + nat_comp['period'].astype(str).map(period_sort)
nat_comp = nat_comp.sort_values('_sort_key_n')

nat_comp_melt = nat_comp.melt(
    id_vars=['time_label_n', '_sort_key_n'],
    value_vars=['pernah_bekerja', 'tidak_pernah_bekerja'],
    var_name='category',
    value_name='count',
)
nat_comp_melt['category'] = nat_comp_melt['category'].map(
    {'pernah_bekerja': 'Pernah Bekerja', 'tidak_pernah_bekerja': 'Tidak Pernah Bekerja'}
)

fig9 = px.line(
    nat_comp_melt.sort_values('_sort_key_n'),
    x='time_label_n',
    y='count',
    color='category',
    category_orders={'time_label_n': time_order},
    markers=True,
    color_discrete_sequence=PLOTLY_COLORS[:2],
    title='National Pernah Bekerja vs. Tidak Pernah Bekerja Totals Over Time (2021–2025)',
    labels={
        'time_label_n': 'Survey Period',
        'count': 'Jumlah Pengangguran (Orang)',
        'category': 'Employment History',
    },
)
fig9.update_layout(
    xaxis_title='Survey Period',
    yaxis_title='Jumlah Pengangguran (Orang)',
    legend_title='Employment History',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig9.write_html(str(OUTPUTS_DIR / 'pernah_vs_tidak_trend.html'))
fig9.show()


### Chart 10: 60+ Age Group — Isolated Annotated Trend

**Why (Q7):** The 60+ cohort exhibits inter-period swings exceeding 100% in 2021–2022, far beyond
any other age group. Isolating it with automatic percentage-change annotations makes the magnitude
of each swing visible and comparable across years. These swings are a data-capture artefact of
how seasonal informal and agricultural labor is classified in each survey wave — not genuine
unemployment cycles — and should always be caveated when reporting 60+ figures.


In [15]:
df60 = df[df['age_group'].astype(str) == '60+'].copy()
df60['time_label_60'] = df60['year'].astype(str) + '-' + df60['period'].astype(str).map(period_abbr)
df60['_sort_key_60'] = df60['year'].astype(int) * 10 + df60['period'].astype(str).map(period_sort)
df60 = df60.sort_values('_sort_key_60').reset_index(drop=True)

# Build annotations for inter-period swings exceeding ±50%
swing_ann = []
for i in range(1, len(df60)):
    prev_val = df60.loc[i - 1, 'jumlah']
    curr_val = df60.loc[i, 'jumlah']
    pct = (curr_val - prev_val) / prev_val * 100
    if abs(pct) > 50:
        swing_ann.append({
            'x': df60.loc[i, 'time_label_60'],
            'y': curr_val,
            'text': f'{pct:+.0f}%',
            'showarrow': True,
            'arrowhead': 2,
            'font': {'size': 10, 'color': 'red'},
            'xref': 'x',
            'yref': 'y',
        })

# Append source note as non-arrow annotation
swing_ann.append({
    'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
    'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'},
})

fig10 = px.line(
    df60,
    x='time_label_60',
    y='jumlah',
    category_orders={'time_label_60': time_order},
    markers=True,
    color_discrete_sequence=[PLOTLY_COLORS[6]],
    title='60+ Age Group: Unemployment Trend with Flagged Inter-Period Swings (2021–2025)',
    labels={'time_label_60': 'Survey Period', 'jumlah': 'Jumlah Pengangguran (Orang)'},
)
fig10.update_layout(
    xaxis_title='Survey Period',
    yaxis_title='Jumlah Pengangguran (Orang)',
    yaxis_tickformat=',',
    annotations=swing_ann,
)
fig10.write_html(str(OUTPUTS_DIR / '60plus_trend.html'))
fig10.show()


### Chart 11: Youth (15–29) Share of National Total Unemployment

**Why (Q8):** Reducing the youth share is the clearest leading indicator of structural labor
market improvement — it shows whether young cohorts are being absorbed faster than the overall
unemployment pool is shrinking. A stable or slowly declining share signals that youth
unemployment is structurally entrenched, not simply moving in lockstep with the national trend.


In [16]:
youth_groups_s = ['15-19', '20-24', '25-29']

nat_total_s = df.groupby(['year', 'period'], observed=True)['jumlah'].sum().reset_index()
youth_total_s = (
    df[df['age_group'].astype(str).isin(youth_groups_s)]
    .groupby(['year', 'period'], observed=True)['jumlah']
    .sum()
    .reset_index()
)
youth_pct = nat_total_s.merge(youth_total_s, on=['year', 'period'], suffixes=('_total', '_youth'))
youth_pct['youth_share_pct'] = youth_pct['jumlah_youth'] / youth_pct['jumlah_total'] * 100
youth_pct['time_label_y'] = youth_pct['year'].astype(str) + '-' + youth_pct['period'].astype(str).map(period_abbr)
youth_pct['_sort_key_y'] = youth_pct['year'].astype(int) * 10 + youth_pct['period'].astype(str).map(period_sort)
youth_pct = youth_pct.sort_values('_sort_key_y')

fig11 = px.line(
    youth_pct,
    x='time_label_y',
    y='youth_share_pct',
    category_orders={'time_label_y': time_order},
    markers=True,
    color_discrete_sequence=[PLOTLY_COLORS[2]],
    title='Youth (15–29) Share of National Total Unemployment (2021–2025)',
    labels={'time_label_y': 'Survey Period', 'youth_share_pct': 'Youth Share (%)'},
)
fig11.update_layout(
    xaxis_title='Survey Period',
    yaxis_title='Youth Share (%)',
    yaxis_ticksuffix='%',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig11.write_html(str(OUTPUTS_DIR / 'youth_share.html'))
fig11.show()


### Chart 12: Seasonal Heatmap — (August − February) / February × 100%

**Why (Q9):** A plain count heatmap (Chart 3) shows magnitude; this percentage-change heatmap
reveals *seasonality*. Positive cells mean August unemployment exceeds February; negative cells
mean the opposite. The 60+ row is expected to show the most extreme values (informal/agricultural
seasonality), while school-leaver cohorts (15–19, 20–24) should show consistent positive spikes
driven by mid-year graduation waves.


In [17]:
feb_data = df[df['period'].astype(str) == 'Februari'].set_index(['year', 'age_group'])['jumlah']
aug_data = df[df['period'].astype(str) == 'Agustus'].set_index(['year', 'age_group'])['jumlah']

seasonal = ((aug_data - feb_data) / feb_data * 100).reset_index()
seasonal.columns = ['year', 'age_group', 'aug_vs_feb_pct']
seasonal['age_group'] = seasonal['age_group'].astype(str)

seasonal_pivot = seasonal.pivot(index='age_group', columns='year', values='aug_vs_feb_pct')
seasonal_pivot = seasonal_pivot.loc[AGE_GROUP_ORDER, sorted(seasonal_pivot.columns)]

fig12 = px.imshow(
    seasonal_pivot,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    text_auto='.1f',
    aspect='auto',
    title='Seasonal Pattern: (August − February) / February × 100 (%) by Age Group & Year',
    labels={'x': 'Year', 'y': 'Age Group', 'color': 'Aug–Feb Change (%)'},
)
fig12.update_layout(
    xaxis_title='Year',
    yaxis_title='Age Group',
    coloraxis_colorbar_title='Aug–Feb (%)',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.12, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig12.write_html(str(OUTPUTS_DIR / 'seasonal_heatmap.html'))
fig12.show()


### Chart 13: Middle-Age (35–54) vs. Youth (15–29) Unemployment Over Time

**Why (Q11):** Middle-age cohorts are expected to be more stable in absolute terms than youth cohorts.
This dual-line comparison reveals whether the magnitude gap between the two cohorts narrows,
widens, or remains fixed over the recovery period — informing whether youth-targeted policy
is outperforming or lagging behind mid-career re-employment support.


In [18]:
youth_groups_c = ['15-19', '20-24', '25-29']
middle_groups_c = ['35-39', '40-44', '45-49', '50-54']

cohort_data = df.copy()
cohort_data['age_group_str'] = cohort_data['age_group'].astype(str)


def _assign_cohort(ag: str) -> str | None:
    if ag in youth_groups_c:
        return 'Youth (15–29)'
    elif ag in middle_groups_c:
        return 'Middle-Age (35–54)'
    return None


cohort_data['cohort'] = cohort_data['age_group_str'].apply(_assign_cohort)
cohort_agg = (
    cohort_data[cohort_data['cohort'].notna()]
    .groupby(['year', 'period', 'cohort'], observed=True)['jumlah']
    .sum()
    .reset_index()
)
cohort_agg['time_label_c'] = cohort_agg['year'].astype(str) + '-' + cohort_agg['period'].astype(str).map(period_abbr)
cohort_agg['_sort_key_c'] = cohort_agg['year'].astype(int) * 10 + cohort_agg['period'].astype(str).map(period_sort)
cohort_agg = cohort_agg.sort_values('_sort_key_c')

fig13 = px.line(
    cohort_agg,
    x='time_label_c',
    y='jumlah',
    color='cohort',
    category_orders={'time_label_c': time_order},
    markers=True,
    color_discrete_sequence=PLOTLY_COLORS[:2],
    title='Middle-Age (35–54) vs. Youth (15–29) Unemployment Over Time (2021–2025)',
    labels={'time_label_c': 'Survey Period', 'jumlah': 'Jumlah Pengangguran (Orang)', 'cohort': 'Cohort'},
)
fig13.update_layout(
    xaxis_title='Survey Period',
    yaxis_title='Jumlah Pengangguran (Orang)',
    legend_title='Cohort',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig13.write_html(str(OUTPUTS_DIR / 'cohort_comparison.html'))
fig13.show()


## Insights & Interpretation

**Chart 1 — Unemployment Trend by Age Group:**
The 20–24 cohort line sits far above all others across every survey period, confirming it
is a structural — not cyclical — feature. The 15–19 cohort shows the largest August spikes
(graduation-driven), while the 60+ line displays unusually large swings tied to how seasonal
informal work is classified in each survey wave. All cohorts declined from 2021 peaks, but
the rate of decline slowed significantly after 2023.

**Chart 2 — February vs. August Gap:**
August totals are consistently higher than February in most years, most visibly in 2021–2022.
The gap is partly driven by the 20–24 and 15–19 cohorts expanding in August (mid-year
graduation waves). By 2024–2025 the gap narrows, consistent with a structural floor being
reached where seasonal flows no longer produce large surpluses.

**Chart 3 — Count Heatmap:**
The 20–24 row is the darkest throughout the grid, confirming its dominance. The 60+ row
shows the largest colour contrast between adjacent columns (Feb/Aug within the same year),
signalling anomalous inter-period volatility. Cohorts 30–54 show gradual lightening from
left to right as recovery progressed.

**Chart 4 — Composition (Pernah vs. Tidak Pernah Bekerja):**
The stacked bars confirm a clear age gradient: `Tidak Pernah Bekerja` (never employed)
dominates the 15–19 and 20–24 stacks (~60–80%), while `Pernah Bekerja` (previously employed)
dominates 35–59. This means youth unemployment is primarily a first-employment access problem,
requiring entry-level job creation and vocational pathways rather than re-employment support.

**Chart 5 — National Year-over-Year Change:**
The 2022 bars (both Feb and Aug) are the most negative, marking the steepest single-year
drop in post-pandemic recovery. By 2024–2025, bars cluster near 0%, confirming that
cyclical recovery momentum has largely exhausted itself.

**Chart 6 — National Total Trend (single line):**
The aggregated line makes the recovery and plateau unmistakable: a steep decline from the
Aug 2021 peak (~9.1 M) through 2023, then a near-flat trajectory in 2024–2025 around
7.3–7.5 M, consistent with the economy hitting a structural unemployment floor.

**Chart 7 — Age Group Ranking per Year:**
The 20–24 bar is the tallest in every single year without exception. The ranking of remaining
cohorts (20–24 > 25–29 > 15–19) is also stable, confirming that the hierarchy is structural.
The bars shorten uniformly across years for most cohorts, but the 20–24 bar shrinks
proportionally less, suggesting youth absorption is the lagging component of recovery.

**Chart 8 — YoY Change by Age Group:**
The 20–24 and 25–29 cohorts dominate in absolute magnitude of YoY change (2022 = steepest
drop). The 60+ cohort shows the most erratic bars, including positive YoY values in some
periods, further flagging its data as seasonality-driven. The 2024–2025 facets show near-zero
bars for almost all cohorts, confirming the structural floor hypothesis at the age-group level.

**Chart 9 — Pernah vs. Tidak Pernah Bekerja National Trend:**
`Pernah Bekerja` (previously employed) falls sharply from 2021 to 2025 — re-employment of
displaced workers is proceeding. `Tidak Pernah Bekerja` (never employed) remains flat
throughout the same period, confirming that the structural first-employment barrier is not
being resolved by the recovery cycle.

**Chart 10 — 60+ Isolated Annotated Trend:**
The auto-flagged annotations show that the Aug 2021 vs. Feb 2021 swing exceeded +100%, and
Aug 2022 vs. Feb 2022 exceeded +60%. By 2023–2025, swings compress below the 50% flag
threshold, suggesting the post-COVID informal labor classifications stabilised. These numbers
should NOT be used to draw policy conclusions about elderly unemployment without further
disaggregation.

**Chart 11 — Youth Share of National Total:**
Youth (15–29) accounts for roughly 50–55% of national unemployment throughout 2021–2025.
The share declines only slowly, confirming that youth absorption is not keeping pace with
the national recovery rate — the structural first-employment problem is persistent.

**Chart 12 — Seasonal Heatmap (Aug−Feb)/Feb%:**
The 60+ row shows cells exceeding +100% in 2021–2022, dwarfing all other cohorts. The 15–19
and 20–24 rows show consistently positive values (August > February), driven by graduation-wave
influx each mid-year. Cohorts 30–54 show near-zero or slightly negative seasonality, consistent
with stable, non-seasonal employment patterns for prime working-age adults.

**Chart 13 — Middle-Age vs. Youth Cohort Comparison:**
Youth (15–29) unemployment is roughly 3–4× larger than middle-age (35–54) unemployment
throughout the study period. Both trend downward from 2021, but the youth line's absolute
decline far exceeds the middle-age decline in raw terms — reflecting the greater volume of
youth displacement that the post-pandemic recovery is addressing.


## Summary / Conclusions

**Key visual insights (13 charts — all 11 analytical questions addressed):**

1. **20–24 structural dominance (Q1, Q7)**: The 20–24 age group is the single largest contributor in every year, persistently 2–3× larger than adjacent cohorts, confirming a structural gap between education completion and labor market absorption.
2. **Post-pandemic recovery, now plateauing (Q2, Q10)**: National total unemployment declined from ~9.1 M (Aug 2021) to ~7.3–7.5 M (2024–2025). The national trend line is essentially flat for the last two years.
3. **February vs. August asymmetry (Q3)**: August totals are systematically higher, driven by mid-year graduation waves in the 15–19 and 20–24 cohorts. The gap narrows in 2024–2025 as the structural floor is approached.
4. **Age-group recovery leaders (Q4)**: The 20–24 and 25–29 cohorts show the largest absolute YoY drops in 2022. The 60+ cohort shows erratic, seasonality-driven bars rather than meaningful trend.
5. **Never-employed is the youth problem (Q5)**: `Tidak Pernah Bekerja` dominates younger cohort stacks; `Pernah Bekerja` dominates older cohort stacks — policy levers differ by age group.
6. **Re-employment outpaces first-time entry (Q6)**: National `Pernah Bekerja` has fallen ~34%; `Tidak Pernah Bekerja` has remained flat — cyclical recovery is mostly over, structural first-employment barriers remain.
7. **60+ anomaly confirmed (Q7)**: Auto-annotated swings exceeding +100% in 2021 are seasonality artefacts, not genuine unemployment cycles. The 60+ cohort requires disaggregation before policy use.
8. **Youth share stable, not declining (Q8)**: Youth (15–29) holds ~50–55% of national unemployment throughout, declining only slowly — youth absorption is not keeping pace with the national recovery.
9. **Seasonal pattern is cohort-specific (Q9)**: The seasonal heatmap confirms that Aug > Feb seasonality is concentrated in 15–19, 20–24, and 60+ cohorts; prime working-age cohorts (30–54) are near-zero seasonal.
10. **Structural floor in 2024–2025 (Q10)**: YoY change bars at the national and age-group levels both converge near 0%; remaining unemployment is structural, not cyclical.
11. **Youth unemployment magnitude vs. middle-age (Q11)**: Youth (15–29) total is 3–4× larger than middle-age (35–54) throughout, and declines faster in absolute terms — but proportion-of-workforce concerns make middle-age a secondary policy priority.

> **Limitations:** All figures are national-level aggregates. Regional, gender, or education-level conclusions cannot be drawn from this dataset. February and August are point-in-time surveys; monthly intra-year trends cannot be inferred. The 60+ cohort's figures are sensitive to informal labor classification and should always be caveated separately.
